In [1]:
import os
from pyspark.sql import SparkSession, functions as F

# Имя каталога
catalog = "lk"

# Доступ к minio
access_key = os.getenv("MINIO_ROOT_USER", "minioadmin")
secret_key = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")
warehouse = os.getenv("LAKEKEEPER_WAREHOUSE", "mydatalab")

#Настройка каталога в Spark
spark = (
    SparkSession.builder.appName("lakekeeper-iceberg-demo")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config(f"spark.sql.catalog.{catalog}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{catalog}.type", "rest")
    .config(f"spark.sql.catalog.{catalog}.uri", "http://127.0.0.1:8181/catalog")
    .config(f"spark.sql.catalog.{catalog}.warehouse", warehouse)
    .config(f"spark.sql.catalog.{catalog}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{catalog}.s3.endpoint", "http://127.0.0.1:9000")
    .config(f"spark.sql.catalog.{catalog}.s3.path-style-access", "true")
    .config(f"spark.sql.catalog.{catalog}.s3.access-key-id", access_key)
    .config(f"spark.sql.catalog.{catalog}.s3.secret-access-key", secret_key)
    .config(f"spark.sql.legacy.parquet.nanosAsLong","true") #Добавили для поддержки Timestamp из паркета
    .config("spark.sql.defaultCatalog", catalog)
    .getOrCreate()
)

#Уровень логирования
spark.sparkContext.setLogLevel("WARN")

In [2]:
#Удаляем таблицы
spark.sql("DROP TABLE IF EXISTS lk.stage.users")
spark.sql("DROP TABLE IF EXISTS lk.stage.payments")
spark.sql("DROP TABLE IF EXISTS lk.stage.trips")
spark.sql("DROP TABLE IF EXISTS lk.stage.events")

DataFrame[]

In [5]:
# Дополнительные Import
from pyspark import SparkFiles #Для локальной файловой системы
from pyspark.sql.functions import current_timestamp #функция текущего времени

In [4]:
# Загрузка users
## Скачиваем файл для обработки
spark.sparkContext.addFile("https://inzhenerka-public.s3.eu-west-1.amazonaws.com/scooters_data_generator/users.parquet")
## Загружаем и обрабатываем файл
users_df = spark.read.parquet('file://' + SparkFiles.get('users.parquet'))
users_df = users_df.withColumn("last_updated", current_timestamp())
## сохраняем фрейм данных в таблицу
users_df.writeTo("lk.stage.users").create()

NameError: name 'current_timestamp' is not defined